In [5]:
import os
import glob
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import imageio
from pathlib import Path
from IPython.display import Image, display

In [6]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class PPOAgent(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)), nn.Tanh(),
            layer_init(nn.Linear(64, 64)),      nn.Tanh(),
            layer_init(nn.Linear(64, action_dim), std=0.01),
        )

    def get_action(self, obs):
        return self.actor_mean(obs)  # deterministic mean


class SACActorNet(nn.Module):
    def __init__(self, obs_dim, action_dim, action_scale, action_bias):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc_mean = nn.Linear(256, action_dim)
        self.fc_logstd = nn.Linear(256, action_dim)
        self.register_buffer("action_scale", action_scale)
        self.register_buffer("action_bias",  action_bias)

    def get_action(self, obs):
        x = F.relu(self.fc1(obs))
        x = F.relu(self.fc2(x))
        return torch.tanh(self.fc_mean(x)) * self.action_scale + self.action_bias

In [7]:
def _detect_alg(run_dir):
    for part in Path(run_dir).parts:
        if part in ("ppo", "sac"):
            return part
    raise ValueError(f"Cannot detect algorithm from path: {run_dir}")


def load_agent(run_dir, device="cpu"):
    """Load PPO or SAC agent from a run directory containing model.pt + config.yaml."""
    cfg = yaml.safe_load(open(os.path.join(run_dir, "config.yaml")))
    env_id  = cfg["env_id"]
    alg     = _detect_alg(run_dir)
    data    = torch.load(os.path.join(run_dir, "model.pt"), map_location=device)

    tmp_env    = gym.make(env_id)
    obs_dim    = int(np.prod(tmp_env.observation_space.shape))
    action_dim = int(np.prod(tmp_env.action_space.shape))

    if alg == "ppo":
        agent   = PPOAgent(obs_dim, action_dim).to(device)
        # strict=False: the saved dict also contains critic/actor_logstd which we don't need for inference
        agent.load_state_dict(data["state_dict"], strict=False)
        obs_rms = data.get("obs_rms")          # dict with 'mean' and 'var', or None
        tmp_env.close()
        return agent, env_id, alg, obs_rms

    else:  # sac
        a_scale = torch.tensor(
            (tmp_env.action_space.high - tmp_env.action_space.low) / 2.0,
            dtype=torch.float32)
        a_bias  = torch.tensor(
            (tmp_env.action_space.high + tmp_env.action_space.low) / 2.0,
            dtype=torch.float32)
        actor = SACActorNet(obs_dim, action_dim, a_scale, a_bias).to(device)
        actor.load_state_dict(data)
        tmp_env.close()
        return actor, env_id, alg, None


def render_episode(agent, env_id, alg, obs_rms=None, device="cpu", max_steps=1000):
    """Roll out one episode and return a list of RGB frames."""
    env = gym.make(env_id, render_mode="rgb_array")
    obs, _ = env.reset()
    frames, total_reward = [], 0.0

    for _ in range(max_steps):
        frames.append(env.render())

        if obs_rms is not None:              # PPO: normalise observation
            obs = (obs - obs_rms["mean"]) / np.sqrt(obs_rms["var"] + 1e-8)
            obs = np.clip(obs, -10, 10)

        obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action = agent.get_action(obs_t).cpu().numpy()[0]

        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    env.close()
    print(f"  Episode return: {total_reward:.2f}  ({len(frames)} frames)")
    return frames


def save_gif_and_display(frames, path, fps=30):
    """Save frames as an animated GIF and display inline."""
    duration_ms = 1000 / fps  # imageio v2.5+ uses duration in ms per frame
    imageio.mimwrite(path, [f.astype(np.uint8) for f in frames], duration=duration_ms, loop=0)
    display(Image(filename=path))


## To render a policy add you run below

In [ ]:
OUTPUT_DIR = "../output"
# OUTPUT_DIR  = ""# add your path to the run you want to visualize
ENV_ID      = "InvertedDoublePendulum-4"   # Change to your environment
DEVICE      = "cpu" # If you have a gpu you can use it
MAX_STEPS   = 1000
FPS         = 60 # Toggle for sped
ALG         = "ppo"
print(f"Output dir : {OUTPUT_DIR}")

Output dir : ../output


In [ ]:
#ppo_run = latest_run(OUTPUT_DIR, ENV_ID, "ppo")
print(f"Loading Agent from: {OUTPUT_DIR}")
agent, env_id, alg, obs_rms = load_agent(OUTPUT_DIR, device=DEVICE)
frames = render_episode(agent, env_id, alg, obs_rms=obs_rms, device=DEVICE, max_steps=MAX_STEPS)
save_gif_and_display(frames, f"{ENV_ID}_{ALG}.gif", fps=FPS)